In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from peft import PeftModel

In [2]:
ft_model_path = "./gptj_mod_mult_clean_final_30p"
model_name = "EleutherAI/gpt-j-6b"

In [3]:
class ModularMultiplicationDataset(Dataset):
    def __init__(self, p=113):
        self.p = p
        self.samples = []
        for a in range(1, p):
            for b in range(1, p):
                target = (a * b) % p
                self.samples.append((a, b, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


In [8]:
def evaluate_gptj_baseline(model):
    p = 113
    batch_size = 32
    device = "cuda"
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    target_token_ids = {
        num: tokenizer.encode(f" {num}", add_special_tokens=False)[0]
        for num in range(1, p)
    }

    dataset = ModularMultiplicationDataset(p=p)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_results = []
    correct_count = 0
    total = len(dataset)

    print(f"Starte Baseline-Evaluierung auf {total} Aufgaben (Z_{p}^x)...")

    model.eval()
    with torch.no_grad():
        for batch_a, batch_b, batch_target in tqdm(dataloader):
            prompts = [
                f"Output ONLY a number. {a.item()} * {b.item()} mod {p}=" 
                for a, b in zip(batch_a, batch_b)
            ]
            
            inputs = tokenizer(prompts, return_tensors="pt", add_special_tokens=False, padding=True).to(device)
            outputs = model(**inputs)
            
            sequence_lengths = inputs.attention_mask.sum(dim=1) - 1
            last_token_logits = outputs.logits[
                torch.arange(outputs.logits.size(0), device=device), 
                sequence_lengths, 
                :
            ]
            pred_token_ids = torch.argmax(last_token_logits, dim=-1)

            target_ids = torch.tensor(
                [target_token_ids[t.item()] for t in batch_target],
                device=device
            )

            correct_mask = (pred_token_ids == target_ids)

            pred_ids_list = pred_token_ids.tolist()
            decoded_preds = [
                tokenizer.decode([token_id]).strip() 
                for token_id in pred_ids_list
            ]

            for a, b, target, pred_text in zip(batch_a, batch_b, batch_target, decoded_preds):
                target_val = target.item()
                try:
                    is_corr = (int(pred_text) == target_val)
                except ValueError:
                    is_corr = False

                if is_corr:
                    correct_count += 1
                
                all_results.append((a.item(), b.item(), target_val, pred_text, is_corr))

    accuracy = (correct_count / total) * 100
    
    print("\n--- ERGEBNIS ---")
    print(f"Gesamtanzahl Aufgaben: {total}")
    print(f"Korrekt gelöst:        {correct_count}")
    print(f"Baseline Accuracy:     {accuracy:.4f} %")

    output_filename = "baseline_evaluation_all_tasks_model_finetuned.txt"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(f"GPT-J Baseline Evaluation (p={p})\n")
        f.write(f"Prompt: Output ONLY a number. {{a}} * {{b}} mod {p} =\n")
        f.write(f"Accuracy: {accuracy:.4f} % ({correct_count}/{total})\n")
        f.write("=" * 60 + "\n")
        f.write(f"{'Aufgabe':<20} | {'Soll':<6} | {'Modell':<10} | {'Status'}\n")
        f.write("-" * 60 + "\n")
        
        for a, b, target, pred_text, is_corr in all_results:
            status = "CORRECT" if is_corr else "WRONG"
            task_str = f"{a} * {b} mod {p}"
            f.write(f"{task_str:<20} | {target:<6} | {pred_text:<10} | {status}\n")

    print(f"Ergebnisse vollständig in '{output_filename}' gespeichert.")

In [16]:
def analyze_results_file(filename="baseline_evaluation_all_tasks_finetuned.txt"):
    total = 0
    repeat_first_count = 0
    single_digit_count = 0
    zero_count = 0
    invalid_count = 0

    with open(filename, "r", encoding="utf-8") as f:
        lines = f.readlines()

    data_started = False
    for line in lines:
        if "---" in line:
            data_started = True
            continue
        if not data_started:
            continue
        
        parts = [p.strip() for p in line.split("|")]
        if len(parts) < 3:
            continue
        
        task_str = parts[0]
        pred_str = parts[2]
        
        try:
            a = int(task_str.split("*")[0].strip())
            pred_val = int(pred_str)
            
            total += 1
            if pred_val == a:
                repeat_first_count += 1
            if 1 <= pred_val <= 9:
                single_digit_count += 1
            if pred_val == 0:
                zero_count += 1
        except (ValueError, IndexError):
            total += 1
            invalid_count += 1
            continue

    if total > 0:
        pct_repeat = (repeat_first_count / total) * 100
        pct_single = (single_digit_count / total) * 100
        pct_zero = (zero_count / total) * 100
        pct_invalid = (invalid_count / total) * 100
        
        print(f"\n--- FEHLERANALYSE DER VORHERSAGEN ---")
        print(f"Gesamt ausgewertet:        {total}")
        print(f"Wiederholt erste Zahl (a): {repeat_first_count} / {total} ({pct_repeat:.2f} %)")
        print(f"Einstellige Zahl (1-9):    {single_digit_count} / {total} ({pct_single:.2f} %)")
        print(f"Vorhersage '0':            {zero_count} / {total} ({pct_zero:.2f} %)")
        print(f"Ungültige/Keine Zahl:      {invalid_count} / {total} ({pct_invalid:.2f} %)")
    else:
        print("Keine Zeilen im erwarteten Format gefunden.")

In [5]:
model_base = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)
model_base.eval()

raw_for_merge = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)
model_finetuned = PeftModel.from_pretrained(raw_for_merge, ft_model_path)
model_finetuned = model_finetuned.merge_and_unload()
model_finetuned.eval()

Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

[transformers] GPTJForCausalLM LOAD REPORT from: EleutherAI/gpt-j-6b
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...27}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...27}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

[transformers] GPTJForCausalLM LOAD REPORT from: EleutherAI/gpt-j-6b
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...27}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...27}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPTJForCausalLM(
  (transformer): GPTJModel(
    (wte): Embedding(50400, 4096)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-27): 28 x GPTJBlock(
        (ln_1): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (attn): GPTJAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): GPTJMLP(
          (fc_in): Linear(in_features=4096, out_features=16384, bias=True)
          (fc_out): Linear(in_features=16384, out_features=4096, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f)

In [7]:
evaluate_gptj_baseline(model_base)

Lade Tokenizer und Modell ...
Starte Baseline-Evaluierung auf 12544 Aufgaben (Z_113^x)...


100%|████████████████████████████████████████| 392/392 [00:23<00:00, 16.56it/s]


--- ERGEBNIS ---
Gesamtanzahl Aufgaben: 12544
Korrekt gelöst:        124
Baseline Accuracy:     0.9885 %
Ergebnisse vollständig in 'baseline_evaluation_all_tasks_model_base.txt' gespeichert.


In [9]:
evaluate_gptj_baseline(model_finetuned)

Lade Tokenizer und Modell ...
Starte Baseline-Evaluierung auf 12544 Aufgaben (Z_113^x)...


100%|████████████████████████████████████████| 392/392 [00:22<00:00, 17.48it/s]


--- ERGEBNIS ---
Gesamtanzahl Aufgaben: 12544
Korrekt gelöst:        5973
Baseline Accuracy:     47.6164 %
Ergebnisse vollständig in 'baseline_evaluation_all_tasks_model_finetuned.txt' gespeichert.


In [19]:
analyze_results_file("baseline_evaluation_all_tasks_model_base.txt")


--- FEHLERANALYSE DER VORHERSAGEN ---
Gesamt ausgewertet:        12544
Wiederholt erste Zahl (a): 9941 / 12544 (79.25 %)
Einstellige Zahl (1-9):    2489 / 12544 (19.84 %)
Vorhersage '0':            56 / 12544 (0.45 %)
Ungültige/Keine Zahl:      42 / 12544 (0.33 %)


In [20]:
analyze_results_file("baseline_evaluation_all_tasks_model_finetuned.txt")


--- FEHLERANALYSE DER VORHERSAGEN ---
Gesamt ausgewertet:        12544
Wiederholt erste Zahl (a): 130 / 12544 (1.04 %)
Einstellige Zahl (1-9):    812 / 12544 (6.47 %)
Vorhersage '0':            2 / 12544 (0.02 %)
Ungültige/Keine Zahl:      1 / 12544 (0.01 %)
